# **Sistemi Lineari: Metodi iterativi**
Consideriamo un sistema lineare nella forma

$$\mathbf{A}\mathbf{x}=\mathbf{b},$$

dove $\mathbf{A}\in\mathbb{R}^{n\times n}$ e $\mathbf{b}\in\mathbb{R}^{n}$ sono noti, mentre $\mathbf{x}\in\mathbb{R}^{n}$ è il vettore incognito.


I medoti di Jacobi, Gauss-Seidel, Richardson, e del gradiente (eccezione fatta per il metodo del gradiente precondizionato) sono accomunati dallo schema iterativo

$$
  \mathbf{x}^{k+1} = \mathbf{B}^k \mathbf{x}^k + \mathbf{g},
$$

con $\mathbf{B}^k = \mathbf{I} - \alpha_k\mathbf{P}^{-1}\mathbf{A}$ e $\mathbf{g} = \mathbf{P}^{-1}\mathbf{b}$.

Segue:

$$
  \mathbf{x}^{k+1} = (\mathbf{I} - \alpha_k\mathbf{P}^{-1}\mathbf{A})\mathbf{x}^k + \mathbf{P}^{-1}\mathbf{b}, \tag{1}
$$

quest'ultima equazione, per una nuova prospettiva, può essere riscritta come

$$
  \mathbf{x}^{k+1} = \mathbf{x}^k + \mathbf{P}^{-1}\mathbf{r}^k,
$$

dove $\mathbf{r}^k$ è il residuo, $\mathbf{r}^k = \mathbf{b}-\mathbf{A}\mathbf{x}^k$.

</br>

**Metodo di Jacobi**: $\mathbf{P} = \text{diag}(\mathbf{A})$, $\alpha_k = 1$.

**Metodo di Gauss-Seidel**: $\mathbf{P} = \text{tril}(\mathbf{A})$, $\alpha_k = 1$.

**Metodo di Richardson stazionario**: $\mathbf{P}$ in base al precondizionatore desiderato, $\alpha_k = \alpha$.

**Metodo del gradiente**: $\mathbf{P}$ in base al precondizionatore desiderato, $\alpha_k = \frac{(\mathbf{d}^k)^T\mathbf{r}^k}{(\mathbf{d}^k)^T\mathbf{A}\mathbf{r}^k}$.


</br></br>

**Nota**: I metodi di Jacobi e Gauss-Seidel vengono tipicamente presentati attraverso la divisione della matrice $\mathbf{A} = \mathbf{D} - \mathbf{E} - \mathbf{F}$, dove $\mathbf{D}$ è la diagonale di $\mathbf{A}$, $-\mathbf{E}$ la triangolare inferiore escludendo la diagonale, $-\mathbf{F}$ la triangolare superiore escludendo la diagonale.  
Lo schema iterativo di Jacobi può essere scritto come
$$
  \mathbf{D}\mathbf{x}^{k+1} = (\mathbf{D}-\mathbf{A})\mathbf{x}^k + \mathbf{b},
$$
ossia
\begin{equation*}
\begin{aligned}
  \mathbf{x}^{k+1} &= \mathbf{D}^{-1}(\mathbf{D}-\mathbf{A})\mathbf{x}^k + \mathbf{D}^{-1}\mathbf{b},\\
   &= \mathbf{D}^{-1}(\mathbf{E}+\mathbf{F})\mathbf{x}^k + \mathbf{D}^{-1}\mathbf{b},
\end{aligned}
\end{equation*}
mentre lo schema di Gauss-Seidel può essere scritto come
$$
  (\mathbf{D} - \mathbf{E})\mathbf{x}^{k+1} = \mathbf{F}\mathbf{x}^k + \mathbf{b},
$$
ossia
$$
  \mathbf{x}^{k+1} = (\mathbf{D} - \mathbf{E})^{-1}\mathbf{F}\mathbf{x}^k + (\mathbf{D} - \mathbf{E})^{-1}\mathbf{b}.
$$


## Parte 1 - Implementazione

In [1]:
# import librerie e funzioni
import numpy as np
# funzione per la risoluzione di sistemi triangolari
from scipy.linalg import solve_triangular
# funzione per gli autovalori
from scipy.linalg import eigvals
from scipy.linalg import eigvalsh
import scipy.linalg

<mark>**Esercizio 1**</mark></br>
Scrivere una function chiamata *jacobi_bg* che, dati $\mathbf{A}$ e $\mathbf{b}$, restituisce la matrice d'iterazione $\mathbf{B}$ ed il vettore di shifting $\mathbf{g}$ associati al metodo di Jacobi. Scrivere quindi una seconda funzione, *gauss_seidel_bg*, che faccia la stessa cosa ma per il metodo di Gauss-Seidel.

Hint: per Jacobi, $\mathbf{P}^{-1}$ è nota in forma chiusa. Per Gauss-Seidel sfruttate la funzione *solve_triangular* del pacchetto *scipy.linalg*.

In [2]:
# Jacobi
def jacobi_bg(A, b=None):
    """
    Metodi di Jacobi
    Input:
    A: matrice quadrata
    b: termine noto (facoltativo)
    Output:
    B: matrice di iterazione di Jacobi
    g: vettore di shifting di Jacobi (se abbiamo come input b)
    """
    n, m = A.shape

    if n != m:
        raise ValueError("Errore: la matrice A non è quadrata\n")
    
    if b is None:
        b = np.zeros(n)
    
    if len(b) != n:
        raise ValueError("Errore: il vettore b non ha la stessa dimensione di A\n")
    
    d = np.diag(A)

    if np.any(d == 0):
        raise ValueError("Errore: la matrice A ha zeri sulla diagonale\n")

    P = np.diag(d)

    # Definisco P^-1, che è possibile dato che P è una matrice diagonale

    P_inv = np.diag(1.0 / d)

    # Costruisco la matrice di iterazione

    B = P_inv @ (P - A)
    g = P_inv @ b

    return B, g

In [3]:
import scipy.linalg

# Gauss-Seidel
def gauss_seidel_bg(A, b=None):
    """
    Metodi di Gauss-Seidel
    Input:
    A: matrice quadrata
    b: termine noto (facoltativo)
    Output:
    B: matrice di iterazione di Gauss-Seidel
    g: vettore di shifting di Gauss-Seidel (se abbiamo come input b)
    """
    n, m = A.shape
    
    if m != n:
        raise ValueError("Errore: la matrice A non è quadrata\n")
    
    if b is None:
        b = np.zeros(n)

    if len(b) != n:
        raise ValueError("Errore: il vettore b non ha la stessa dimensione di A\n")
    
    M = np.tril(A)
    U = -np.triu(A, k=1)

    B = scipy.linalg.solve_triangular(M, U, lower=True)
    g = scipy.linalg.solve_triangular(M, b, lower=True)
  
    return B, g

<mark>**Esercizio 2**</mark></br>
Scrivere una function chiamata *iterative_solve* che, dati

- $\mathbf{A}$ matrice del sistema
- $\mathbf{b}$ termine noto
- $\mathbf{x}_{0}$ guess iniziale
- $\mathbf{B}$ matrice di iterazione
- $\mathbf{g}$ vettore di shifting

approssimi la soluzione $\mathbf{x}$ con il metodo iterativo corrispondente. La function dovrà accettare anche altri due parametri: **nmax**, cioè il numero massimo di iterazioni, **rtoll**, la tolleranza relativa richiesta. Il particolare, il metodo iterativo va arrestato se

$$\frac{\|\mathbf{r}^{(k)}\|}{\|\mathbf{b}\|}<\textbf{rtoll},$$

dove $\mathbf{r}^{(k)}:= \mathbf{b} - \mathbf{A}\mathbf{x}^{(k)}$ è il residuo alla *k*-esima iterazione.

*Nota*: costruite la function di modo che, in output, essa restituisca la lista delle iterate $[\mathbf{x}_{0},\dots,\mathbf{x}_{N}]$.

In [4]:
def iterative_solve(A, b, x0, B, g, nmax, rtoll):
    """
    Metodo di risoluzione utilizzando i metodi iterativi

    Input:
    A: matrice quadrata
    b: termine noto
    x0: vettore di innesco
    B: matrice di iterazione
    g: vettore di shifting
    nmax: numero massimo di iterazioni
    rtoll: tolleranza relativa richiesta

    Output:
    xiter: lista contenente tutte le iterate

    """
    # 1. TRASFORMAZIONE E TIPO DI DATO
    try:
        A = np.array(A, dtype=float)
        b = np.array(b, dtype=float)
        x0 = np.array(x0, dtype=float)
        B = np.array(B, dtype=float)
        g = np.array(g, dtype=float)
    except (ValueError, TypeError):
        raise TypeError("Tutti gli input (A, b, x0, B, g) devono contenere solo numeri.")

    # 2. CONTROLLO DIMENSIONALITÀ (Matrici vs Vettori)
    if A.ndim != 2:
        raise ValueError(f"A deve essere una matrice (2D), invece ha dimensione {A.ndim}D.")
    if B.ndim != 2:
        raise ValueError(f"B deve essere una matrice (2D), invece ha dimensione {B.ndim}D.")
    if b.ndim != 1:
        raise ValueError(f"b deve essere un vettore (1D), invece ha dimensione {b.ndim}D.")
    
    # 3. CONTROLLO COERENZA (Shape)
    n = A.shape[0]
    
    if A.shape[1] != n:
        raise ValueError(f"La matrice A deve essere quadrata. Ricevuta: {A.shape[0]}x{A.shape[1]}.")
    
    if B.shape != (n, n):
        raise ValueError(f"La matrice B deve essere quadrata e della stessa taglia di A ({n}x{n}).")
    
    if b.shape[0] != n:
        raise ValueError(f"Il vettore b ha lunghezza {b.shape[0]}, ma deve essere {n}.")
        
    if x0.shape[0] != n:
        raise ValueError(f"Il vettore x0 ha lunghezza {x0.shape[0]}, ma deve essere {n}.")
        
    if g.shape[0] != n:
        raise ValueError(f"Il vettore g ha lunghezza {g.shape[0]}, ma deve essere {n}.")

    # 4. CONTROLLO PARAMETRI SCALARI
    if nmax <= 0:
        raise ValueError("nmax deve essere un intero positivo.")
    if rtoll <= 0:
        raise ValueError("rtoll deve essere un numero positivo.")


    xiter = [x0]    # vettore delle iterate inizializzata con il primo valore x0
    x0 = xold
    bnorm = np.linalg.norm(b)

    if bnorm < 1e-16:   # se la norma di b è molto piccola potrei avere problemi nel calcolo del residuo relativo (in cui divido per bnorm) 
        bnorm = 1.0     # --> impostanto bnorm = 1.0 vado a calcolare non più il residuo relativo, ma quello assoluto --> ||r_k|| < rtoll

    r0 = np.linalg.norm(b - A @ x0) / bnorm     # calcolo il primo residuo

    r = [r0]  # vettore dei residui inizializzato con il primo valore r0

    i = 0      # contatore iterazione

    # ciclo iterativo
    while r[i] > rtoll and i <  nmax:
        xnew = B @ xold + g
        xold = xnew
        xiter.append(xnew)
        residuo_attuale = np.linalg.norm(b - A @ xnew) / bnorm
        r.append(residuo_attuale)
        i += 1
    return np.array(xiter), np.array(r)

## Parte 2 - Sperimentazione

<mark>**Esercizio 3**</mark></br>
Si consideri la seguente matrice quadrata


$$\mathbf{A}=\left[\begin{array}{cccccc}
1 & 1 & 1 & 1 & \dots & 1\\
R_{1} & - R_{2} & 0 & 0 & \dots & 0\\
0 & R_{1} & - R_{2} & 0 &  \dots & 0\\
\dots & 0 & \ddots & \ddots &   & \dots\\
\dots & \dots &  & \ddots &  \ddots & \dots\\
0 & 0 & 0 & \dots & R_{1} & - R_{2} \\
\end{array}\right]$$


di dimensione $n=100$, avendo posto $R_{1}=1$ ed $R_{2}=2$.
</br></br>

a) Assemblare le matrici di iterazione $B_{\text{J}}$ e $B_{\text{GS}}$ dei metodi di Jacobi e Gauss-Seidel, quindi calcolarne i rispettivi raggi spettrali. La condizione
necessaria e sufficiente per la convergenza del metodo iterativo è soddisfatta in entrambi i casi?

*Hint: Usare la function $\texttt{eigvals}$ di $\texttt{scipy.linalg}$ per il calcolo degli autovalori.*
</br></br>

b) Sia $\mathbf{b}=[2,1,1,\dots,1]^{\top}\in\mathbb{R}^{n}$. Approssimare la soluzione del sistema lineare $\mathbf{A}\mathbf{x}=\mathbf{b}$ con il metodo di Jacobi. Si pongano $$\mathbf{x}_{0}=[0,\dots,0]^{\top},\quad\texttt{rtoll}=10^{-6},\quad\texttt{nmax}=1000.$$ Il metodo converge? Se sì, in quante iterazioni?

In [5]:
# Costruzione della matrice A

n = 100

R2 = 2*np.ones(n)
R1 = np.ones(n-1)

A = -np.diag(R2) + np.diag(R1, k=-1)
A[0,:] = 1.0



In [6]:
# a) Raggi spettrali

# calcolo le matrici di iterazione di Jacobi e GS

B_gs, g_gs = gauss_seidel_bg(A)
B_j, g_j = jacobi_bg(A)

# calcolo i raggi spettrali

rho_gs = np.max(np.abs(scipy.linalg.eigvals(B_gs)))
rho_j = np.max(np.abs(scipy.linalg.eigvals(B_j)))

# Stampo i raggi spettrali di Jacobi e Gauss-Seidel

if rho_gs < 1:
    print(f"Il raggio spettrale della matrice di Gauss-Seidel è: {rho_gs}, il metodo converge\n")
else:
    print(f"Il raggio spettrale della matrice di Gauss-Seidel è: {rho_gs}, il metodo NON converge\n")

if rho_j < 1:
    print(f"Il raggio spettrale della matrice di Jacobi è: {rho_j}, il metodo converge\n")
else:
    print(f"Il raggio spettrale della matrice di Jacobi è: {rho_j}, il metodo NON converge\n")

Il raggio spettrale della matrice di Gauss-Seidel è: 0.9999999999999998, il metodo converge

Il raggio spettrale della matrice di Jacobi è: 0.7071067811865475, il metodo converge



In [ ]:
# b)
#  termine noto

# Guess iniziale
x0 = np.zeros(n)

# Applicazione di Jacobi


# Numero di iterazioni


<mark>**Esercizio 4**</mark></br>
Si considerino la matrice e il termine noto
</br>
</br>
\begin{equation*}
\mathbf{A}=\left[\begin{array}{rrrrrrr}
9 & -3 & 1 &  &  & & \\
-3 & 9 & -3 & 1 &  & & \\
1 & -3 & 9 & -3 & 1 & & \\
& 1 & -3 & 9 & -3 & 1 &\\
& & 1 & -3 & 9 & -3 & 1 \\
& & & 1 & -3 & 9 & -3  \\
& & & & 1 & -3 & 9  \\
\end{array}\right],\quad\quad \mathbf{b}=\left[\begin{array}{c}7\\4\\5\\5\\5\\4\\7\end{array}\right].
\end{equation*}
</br>
a) Studiare le caratteristiche della matrice $\mathbf{A}$, stabilendo se sia simmetrica, definita positiva e a dominanza diagonale per righe $^*$.
</br></br>
b) Approssimare la soluzione del sistema lineare $\mathbf{A}\mathbf{x}=\mathbf{b}$ con i metodi di Jacobi e di Gauss-Seidel, utilizzando il vettore nullo come guess iniziale. Si pongano $$\texttt{rtoll}=10^{-6} \quad \text{and} \quad \texttt{nmax}=1000.$$ 
Confrontare il numero
di iterazioni necessarie per arrivare a convergenza per i due metodi e commentare i risultati
ottenuti.
</br>
</br>
La soluzione esatta del sistema lineare è $\mathbf{x}=[1, 1, 1, 1, 1, 1, 1]^T$.
</br>
</br>
</br>
</br>
$^*$*Hint: sfruttare la function $\texttt{eigvalsh}$ del pacchetto $\texttt{scipy.linalg}$, che restituisce gli autovalori per una matrice simmetrica o Hermitiana.*

In [ ]:
# a) Assemblaggio di A e check delle proprietà

n = 7
A = -3 * np.diag(np.ones(n-1), 1) + np.diag(np.ones(n-2), 2)
A = 9 * np.diag(np.ones(n)) + A + A.T


In [ ]:
# b) Applicazione dei metodi e confronto
b = np.array([7, 4, 5, 5, 5, 4, 7])
# guess iniziale
x0 = np.zeros(n)
# risolvo Jacobi e Guass-Seidel


In [ ]:
# Tabella che mi stampa la convergenza e il numero di iterazioni
# print("\t\t\tJacobi\tGauss-Seidel\n" + "-"*44)
# print("Convergenza:\t\t%s\t%s" % (len(xj)-1 < 1000, len(xgs)-1 < 1000))
# print("Numero di iterazioni:\t%d\t%d" % (len(xj)-1, len(xgs)-1))

## Parte 3 - Metodi pre-implementati: gradiente e gradiente coniugato

<mark>**Esercizio 5**</mark></br>
La function $\texttt{cg}$ del pacchetto $\texttt{scipy.sparse.linalg}$ implementa il metodo del gradiente coniugato. Viceversa, la function $\texttt{gdescent}$, disponibile nello script $\texttt{utils.py}$, implementa il metodo del gradiente.
</br></br>
Una volta appurato che entrambi i metodi sono applicabili al problema dell'esercizio 5
</br></br>
a) Approssimare la soluzione del sistema con i metodi del gradiente e del gradiente coniugato. Si utilizzino gli stessi iperparametri usati all'es. 5 (guess iniziale, tolleranza relativa, numero massimo di iterazioni). I metodi convergono? Che soluzione si ottiene?
</br></br>
b) Nei due casi, quante iterazioni ci sono volute? *Hint: per $\texttt{cg}$, sfruttate l'input opzionale $\texttt{callback}$*!

In [ ]:


# numero iterazioni metodo del gradiente
# print("Numero di iterazioni effettuate: %d" % (len(xg)-1))

In [ ]:
from scipy.sparse.linalg import cg

# Definisco la lista con la guess iniziale
xcg = [x0]
# chiamo la function cg del gradiente coniugato
x, info = cg(A, b, x0, rtol=1e-6, maxiter=1000,
             callback=lambda xk: xcg.append(xk+0.0))

print("\nSoluzione approssimata (CG):")
print(x)
print("Numero di iterazioni effettuate: %d" % (len(xcg)-1))

## Esercizi per casa

<mark>**Esercizio 6**</mark></br>
Scrivete le seguenti function a valori booleani (vero o falso):

- **sym** che, data $\mathbf{A}$, restituisce $\texttt{True}$ se e solo se $\mathbf{A}$ è simmetrica;

- **sdp** che, data $\mathbf{A}$, restituisce $\texttt{True}$ se e solo se $\mathbf{A}$ è simmetrica definita positiva;

- **rowdom** che, data $\mathbf{A}$, restituisce $\texttt{True}$ se e solo se $\mathbf{A}$ è a dominanza diagonale per righe.

In [ ]:
#def sym(A):


#def sdp(A):


#def rowdom(A):


<mark>**Esercizio 7**</mark></br>
Si considerino la matrice pentadiagonale ed il termine noto
\begin{equation*}
\mathbf{A}=\left[\begin{array}{rrrrrrr}
5 & -1 & -1 &  &  & & \\
-1 & 5 & -1 & -1 &  & & \\
-1 & -1 & 5 & -1 & -1 & & \\
& \ddots  & \ddots & \ddots & \ddots & \ddots  &\\
& & -1 & -1 & 5 & -1 & -1 \\
& & &  -1 & -1 & 5 & -1 \\
& & & & -1 & -1 & 5 \\
\end{array}\right],\quad\quad \mathbf{b}=\left[\begin{array}{c}0.2\\0.2\\0.2\\\vdots\\0.2\\0.2\\0.2\end{array}\right].
\end{equation*}
</br>
a) La matrice $\mathbf{A}$ è simmetrica definitiva positiva?
</br></br>
b) Approssimare la soluzione del sistema lineare con i metodi di Jacobi, Gauss-Seidel, Gradiente e Gradiente Coniugato (si utilizzi il vettore nullo come guess iniziale, $10^{-5}$ come tolleranza relativa, $1000$ come numero massimo di iterazioni).
</br></br>
c) Plottare l'andamento del residuo relativo $\|\mathbf{r}^{(k)}\|/\|\mathbf{b}\|$ in funzione delle iterate $k$, mettendo così a paragone i quattro metodi.


In [ ]:
# a) Assemblaggio matrice e check proprietà


In [ ]:
# b) Applicazione dei metodi iterativi


# Gradiente

# Gradiente coniugato (sfruttiamo il callback per salvare le iterate: ci servirà dopo)

# Jacobi

# Gauss-Seidel


In [ ]:
# c) Calcolo dei residui e plot


In [ ]:
import matplotlib.pyplot as plt

# Plot in scala logaritmica sulle y
